In [46]:
import pandas as pd
import numpy as np
import random
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io
import logging
from datetime import datetime, timedelta

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')



### 1. Carregar o Dataset

In [83]:
url = "https://www.bcb.gov.br/pda/desig/desenrola/dados_desenrola.csv"
local_file = "../data/raw/dados_desenrola.csv"
csv_config_br = {"sep": ";", "decimal": ",", "encoding": "utf-8"}

def extract_data(url: str, local_file: str, csv_config: dict) -> pd.DataFrame:
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        logging.info("Tentando baixar o dataset online...")
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        logging.info("Download bem-sucedido. Carregando os dados...")
        df = pd.read_csv(
            io.StringIO(response.text),
            **csv_config
            )
        logging.info(f"Dados extraídos com sucesso da internet. [{len(df)}] registros encontrados.")
    except Exception as e:
        logging.error(f"Não foi possível baixar da internet. Motivo: {e}")
        logging.info("Utilizando arquivo local como alternativa...")
        df = pd.read_csv(
            local_file,
            **csv_config
            )
        logging.info(f"Dados extraídos com sucesso de arquivo local. [{len(df)}] registros encontrados.")
    return df

df_raw = extract_data(url, local_file, csv_config_br)
df_raw.info()


INFO - Tentando baixar o dataset online...
INFO - Download bem-sucedido. Carregando os dados...
INFO - Dados extraídos com sucesso da internet. [10598] registros encontrados.


<class 'pandas.DataFrame'>
RangeIndex: 10598 entries, 0 to 10597
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   DATA_BASE                     10598 non-null  int64  
 1   TIPO_DESENROLA                10598 non-null  int64  
 2   UNIDADE_FEDERACAO             10598 non-null  str    
 3   COD_CONGLOMERADO_FINANCEIRO   10598 non-null  int64  
 4   NOME_CONGLOMERADO_FINANCEIRO  10598 non-null  str    
 5   NUMERO_OPERACOES              10598 non-null  int64  
 6   VOLUME_OPERACOES              10598 non-null  float64
dtypes: float64(1), int64(4), str(2)
memory usage: 579.7 KB


#### 1.5 Sujar o Dataset para limpeza posterior

In [84]:
def dirty_data(df, noise_level=0.02):
    # Cria uma cópia "suja" de parte do dataframe original (preservando os dados limpos) e adiciona essa cópia ao final do dataframe original.
    df_clean = df.copy()
    corrupted_rows = []
    
    # Introduz valores nulos
    df_nulls = df_clean.sample(frac=noise_level).copy()
    for col in df_nulls.columns:
        df_nulls.loc[df_nulls.sample(frac=0.5).index, col] = np.nan
    corrupted_rows.append(df_nulls)
    
    # Introduz espaços em branco em colunas de texto
    df_text_columns = df_clean.select_dtypes(include=['object', 'string']).columns
    if not df_text_columns.empty:
        df_spaces = df_clean.sample(frac=noise_level).copy()
        for col in df_text_columns:
            df_spaces[col] = '   ' + df_spaces[col].astype(str) + '   '
        corrupted_rows.append(df_spaces)
        
    # Introduz datas inválidas
    df_dates = df_clean.sample(frac=noise_level).copy()
    df_dates['DATA_BASE'] = 'invalid_date'
    corrupted_rows.append(df_dates)
        
    # Introduz outliers em colunas numéricas
    numeric_columns = df_clean.select_dtypes(include=np.number).columns
    if not numeric_columns.empty:
        df_outliers = df_clean.sample(frac=noise_level).copy()
        for column in numeric_columns:
            df_outliers[column] = df_clean[column].max() * 10
        corrupted_rows.append(df_outliers)
        
    # Introduz duplicatas
    df_duplicates = df_clean.sample(frac=noise_level).copy()
    corrupted_rows.append(df_duplicates)
    
    df_dirty = pd.concat([df_clean] + corrupted_rows, ignore_index=True)

    # Converte as colunas de inteiros para Int64 para permitir valores nulos, mantendo a consistência de tipo com o dataframe original.
    int_cols = df_clean.select_dtypes(include='int64').columns
    int_cols = [col for col in int_cols if col != 'DATA_BASE'] # Filtra e remove a coluna de data
    for col in int_cols:
        df_dirty[col] = df_dirty[col].astype('Int64')

    return df_dirty

df_dirty = dirty_data(df_raw)
os.makedirs("../data/raw", exist_ok=True)
df_dirty.to_csv("../data/raw/desenrola_sujo.csv", index=False)
print(f"Dataset gerado com {len(df_dirty)} registros.")
print(df_dirty.tail())


Dataset gerado com 11658 registros.
      DATA_BASE  TIPO_DESENROLA UNIDADE_FEDERACAO  \
11653    202506               3                GO   
11654    202509               1                PE   
11655    202503               1                PE   
11656    202407               3                MS   
11657    202407               1                SP   

       COD_CONGLOMERADO_FINANCEIRO NOME_CONGLOMERADO_FINANCEIRO  \
11653                        80075        BRADESCO - PRUDENCIAL   
11654                        80996           INTER - PRUDENCIAL   
11655                        80336     BTG PACTUAL - PRUDENCIAL   
11656                        30379                    SANTANDER   
11657                        49906                           BB   

       NUMERO_OPERACOES  VOLUME_OPERACOES  
11653                26        2648011.41  
11654                 2            328.71  
11655                 1             76.12  
11656                59         756639.64  
11657                1

### 2. Inspecionar o Dataset

In [86]:
def inspect_data(df):
    print ("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    print (f"Shape: {df.shape}")
    print (f"\nColunas: {list(df.columns)}")
    print (f"\nTipos de dados:\n{df.dtypes}")
    print (f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print (f"\nPrimeiros registros:\n{df.head()}")
    print (f"\nEstatísticas descritivas:\n{df.describe()}")
    return df.describe(include= "all" )

inspect_data(df_dirty)


=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (11658, 7)

Colunas: ['DATA_BASE', 'TIPO_DESENROLA', 'UNIDADE_FEDERACAO', 'COD_CONGLOMERADO_FINANCEIRO', 'NOME_CONGLOMERADO_FINANCEIRO', 'NUMERO_OPERACOES', 'VOLUME_OPERACOES']

Tipos de dados:
DATA_BASE                        object
TIPO_DESENROLA                    Int64
UNIDADE_FEDERACAO                   str
COD_CONGLOMERADO_FINANCEIRO       Int64
NOME_CONGLOMERADO_FINANCEIRO        str
NUMERO_OPERACOES                  Int64
VOLUME_OPERACOES                float64
dtype: object

Valores nulos por coluna:
DATA_BASE                       106
TIPO_DESENROLA                  106
UNIDADE_FEDERACAO               106
COD_CONGLOMERADO_FINANCEIRO     106
NOME_CONGLOMERADO_FINANCEIRO    106
NUMERO_OPERACOES                106
VOLUME_OPERACOES                106
dtype: int64

Primeiros registros:
  DATA_BASE  TIPO_DESENROLA UNIDADE_FEDERACAO  COD_CONGLOMERADO_FINANCEIRO  \
0    202309               2                AC                        49906  

,DATA_BASE,TIPO_DESENROLA,UNIDADE_FEDERACAO,COD_CONGLOMERADO_FINANCEIRO,NOME_CONGLOMERADO_FINANCEIRO,NUMERO_OPERACOES,VOLUME_OPERACOES
count,11552.0,11552.0,11552,11552.0,11552,11552.0,1.155200e+04
unique,34.0,<NA>,54,<NA>,115,<NA>,NaN
top,202311.0,<NA>,SP,<NA>,BRADESCO,<NA>,NaN
freq,727.0,<NA>,629,<NA>,976,<NA>,NaN
mean,NaN,2.056614,NaN,16939360.676073,NaN,8772.651922,4.021623e+07
std,NaN,3.88557,NaN,116629204.268123,NaN,62246.061436,2.888920e+08
min,NaN,1.0,NaN,10045.0,NaN,1.0,1.000000e-02
25%,NaN,1.0,NaN,30379.0,NaN,2.0,1.183085e+03
50%,NaN,1.0,NaN,51750.0,NaN,9.0,1.865328e+04
75%,NaN,2.0,NaN,80185.0,NaN,89.0,2.933880e+05


### 03. Limpar e Tratar os Dados